<a href="https://colab.research.google.com/github/divyadharshiniug/24ADI003L_24BAD023/blob/main/Cloud_Storage_Parquet_Catalog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
from pathlib import Path

# Simulated cloud object-storage bucket
BUCKET = Path("/content/cloud_bucket")

# Folders similar to a cloud data lake
RAW = BUCKET / "raw"
PROCESSED = BUCKET / "processed"
CATALOG = BUCKET / "catalog"

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
CATALOG.mkdir(parents=True, exist_ok=True)

# Create a simple dataset
data = {
    "student_id": [101, 102, 103, 104, 105, 106],
    "student_name": [
        "Anu",
        "Bala",
        "Charan",
        "Divya",
        "Ezhil",
        "Farhan"
    ],
    "department": [
        "AI&DS",
        "CSE",
        "AI&DS",
        "ECE",
        "CSE",
        "AI&DS"
    ],
    "marks": [82, 76, 91, 68, 88, 79]
}

df = pd.DataFrame(data)

# Save the dataset locally
local_file = Path("/content/students.csv")
df.to_csv(local_file, index=False)

print("Sample dataset created successfully.")
print(df)

Sample dataset created successfully.
   student_id student_name department  marks
0         101          Anu      AI&DS     82
1         102         Bala        CSE     76
2         103       Charan      AI&DS     91
3         104        Divya        ECE     68
4         105        Ezhil        CSE     88
5         106       Farhan      AI&DS     79


In [ ]:
# Upload the CSV file into the simulated cloud bucket
!cp /content/students.csv /content/cloud_bucket/raw/

!echo "File uploaded successfully."

# List files in the raw-data folder
!ls -lh /content/cloud_bucket/raw/

# Display the contents of the uploaded file
!cat /content/cloud_bucket/raw/students.csv

File uploaded successfully.
total 4.0K
-rw-r--r-- 1 root root 148 Aug 11 04:21 students.csv
student_id,student_name,department,marks
101,Anu,AI&DS,82
102,Bala,CSE,76
103,Charan,AI&DS,91
104,Divya,ECE,68
105,Ezhil,CSE,88
106,Farhan,AI&DS,79


In [ ]:
import shutil
from pathlib import Path


class SimpleCloudStorage:
    def __init__(self, bucket_path):
        self.bucket_path = Path(bucket_path)

    def upload(self, local_file, destination_folder):
        source = Path(local_file)
        destination = self.bucket_path / destination_folder / source.name

        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

        print(f"Uploaded: {destination}")

    def list_files(self, folder=""):
        location = self.bucket_path / folder

        print(f"Files inside: {location}")

        for file in location.rglob("*"):
            if file.is_file():
                print(file.relative_to(self.bucket_path))

    def download(self, cloud_file, local_destination):
        source = self.bucket_path / cloud_file
        destination = Path(local_destination)

        shutil.copy2(source, destination)

        print(f"Downloaded to: {destination}")

    def copy_file(self, source_file, destination_file):
        source = self.bucket_path / source_file
        destination = self.bucket_path / destination_file

        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

        print(f"Copied to: {destination}")

    def delete_file(self, cloud_file):
        file_path = self.bucket_path / cloud_file

        if file_path.exists():
            file_path.unlink()
            print(f"Deleted: {file_path}")
        else:
            print("File not found.")


storage = SimpleCloudStorage("/content/cloud_bucket")

print("Listing uploaded files:")
storage.list_files()

Listing uploaded files:
Files inside: /content/cloud_bucket
raw/students.csv


In [ ]:
import pandas as pd

csv_path = "/content/cloud_bucket/raw/students.csv"

cloud_df = pd.read_csv(csv_path)

print("Data read from simulated cloud storage:")
display(cloud_df)

Data read from simulated cloud storage:


,student_id,student_name,department,marks
0,101,Anu,AI&DS,82
1,102,Bala,CSE,76
2,103,Charan,AI&DS,91
3,104,Divya,ECE,68
4,105,Ezhil,CSE,88
5,106,Farhan,AI&DS,79


In [ ]:
# Install PyArrow if required
!pip install -q pyarrow

import pandas as pd

csv_path = "/content/cloud_bucket/raw/students.csv"
parquet_path = "/content/cloud_bucket/processed/students.parquet"

df = pd.read_csv(csv_path)

df.to_parquet(
    parquet_path,
    engine="pyarrow",
    compression="snappy",
    index=False
)

print("CSV file converted into Parquet successfully.")
print("Parquet location:", parquet_path)

CSV file converted into Parquet successfully.
Parquet location: /content/cloud_bucket/processed/students.parquet


In [ ]:
import os

csv_size = os.path.getsize(
    "/content/cloud_bucket/raw/students.csv"
)

parquet_size = os.path.getsize(
    "/content/cloud_bucket/processed/students.parquet"
)

print("CSV file size:", csv_size, "bytes")
print("Parquet file size:", parquet_size, "bytes")

CSV file size: 148 bytes
Parquet file size: 2877 bytes


In [ ]:
import pandas as pd

parquet_df = pd.read_parquet(
    "/content/cloud_bucket/processed/students.parquet",
    engine="pyarrow"
)

print("Data read from Parquet file:")
display(parquet_df)

Data read from Parquet file:


,student_id,student_name,department,marks
0,101,Anu,AI&DS,82
1,102,Bala,CSE,76
2,103,Charan,AI&DS,91
3,104,Divya,ECE,68
4,105,Ezhil,CSE,88
5,106,Farhan,AI&DS,79


In [ ]:
department_summary = (
    parquet_df
    .groupby("department", as_index=False)
    .agg(
        student_count=("student_id", "count"),
        average_marks=("marks", "mean"),
        maximum_marks=("marks", "max")
    )
)

department_summary["average_marks"] = (
    department_summary["average_marks"].round(2)
)

print("Department-wise analysis:")
display(department_summary)

Department-wise analysis:


,department,student_count,average_marks,maximum_marks
0,AI&DS,3,84.0,91
1,CSE,2,82.0,88
2,ECE,1,68.0,68


In [ ]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(
    "/content/cloud_bucket/processed/students.parquet"
)

print("Parquet Schema:")
print(parquet_file.schema)

print("\nNumber of rows:")
print(parquet_file.metadata.num_rows)

print("\nNumber of columns:")
print(parquet_file.metadata.num_columns)

print("\nNumber of row groups:")
print(parquet_file.metadata.num_row_groups)

Parquet Schema:
required group field_id=-1 schema {
  optional int64 field_id=-1 student_id;
  optional binary field_id=-1 student_name (String);
  optional binary field_id=-1 department (String);
  optional int64 field_id=-1 marks;
}


Number of rows:
6

Number of columns:
4

Number of row groups:
1


In [ ]:
import os
import pandas as pd
from pathlib import Path
from datetime import datetime

BUCKET = Path("/content/cloud_bucket")

catalog_records = []

for file_path in BUCKET.rglob("*"):

    if not file_path.is_file():
        continue

    file_type = file_path.suffix.lower().replace(".", "")
    row_count = None
    column_count = None
    column_names = None

    try:
        if file_type == "csv":
            temp_df = pd.read_csv(file_path)

            row_count = len(temp_df)
            column_count = len(temp_df.columns)
            column_names = ", ".join(temp_df.columns)

        elif file_type == "parquet":
            temp_df = pd.read_parquet(file_path)

            row_count = len(temp_df)
            column_count = len(temp_df.columns)
            column_names = ", ".join(temp_df.columns)

    except Exception as error:
        print(f"Could not inspect {file_path.name}: {error}")

    catalog_records.append(
        {
            "file_name": file_path.name,
            "relative_path": str(file_path.relative_to(BUCKET)),
            "file_type": file_type,
            "size_bytes": file_path.stat().st_size,
            "row_count": row_count,
            "column_count": column_count,
            "column_names": column_names,
            "last_modified": datetime.fromtimestamp(
                file_path.stat().st_mtime
            ).strftime("%Y-%m-%d %H:%M:%S")
        }
    )

catalog_df = pd.DataFrame(catalog_records)

print("Data catalog created successfully.")
display(catalog_df)

Data catalog created successfully.


,file_name,relative_path,file_type,size_bytes,row_count,column_count,column_names,last_modified
0,students.parquet,processed/students.parquet,parquet,2877,6,4,"student_id, student_name, department, marks",2026-08-11 04:22:02
1,students.csv,raw/students.csv,csv,148,6,4,"student_id, student_name, department, marks",2026-08-11 04:21:49


In [ ]:
catalog_csv = "/content/cloud_bucket/catalog/data_catalog.csv"
catalog_parquet = "/content/cloud_bucket/catalog/data_catalog.parquet"

catalog_df.to_csv(
    catalog_csv,
    index=False
)

catalog_df.to_parquet(
    catalog_parquet,
    index=False
)

print("Catalog saved successfully.")
print("CSV catalog:", catalog_csv)
print("Parquet catalog:", catalog_parquet)

Catalog saved successfully.
CSV catalog: /content/cloud_bucket/catalog/data_catalog.csv
Parquet catalog: /content/cloud_bucket/catalog/data_catalog.parquet


In [ ]:
!find /content/cloud_bucket -type f

/content/cloud_bucket/processed/students.parquet
/content/cloud_bucket/raw/students.csv
/content/cloud_bucket/catalog/data_catalog.csv
/content/cloud_bucket/catalog/data_catalog.parquet


In [ ]:
search_format = "parquet"

result = catalog_df[
    catalog_df["file_type"] == search_format
]

print("Parquet files available in the catalog:")
display(result)

Parquet files available in the catalog:


,file_name,relative_path,file_type,size_bytes,row_count,column_count,column_names,last_modified
0,students.parquet,processed/students.parquet,parquet,2877,6,4,"student_id, student_name, department, marks",2026-08-11 04:22:02
